<a href="https://colab.research.google.com/github/ibrahimbarghout/robust-ecg-domain-generalization/blob/main/notebooks/03_dataset_exploration.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [70]:
# PTB-XL Research Project
## 3. Dataset Exploration

#This notebook explores the structure, labels, demographics,
#recording characteristics, rhythm annotations, and signal-quality
#annotations of the PTB-XL dataset.

In [71]:
from google.colab import drive

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [72]:
import os
import pandas as pd
import numpy as np

from ast import literal_eval
from collections import Counter

In [73]:
PROJECT_PATH = "/content/drive/MyDrive/PTB-XL Research Project"

DATA_PATH = os.path.join(
    PROJECT_PATH,
    "data",
    "ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3"
)

print("Dataset path:")
print(DATA_PATH)

Dataset path:
/content/drive/MyDrive/PTB-XL Research Project/data/ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3


In [74]:
df = pd.read_csv(
    os.path.join(DATA_PATH, "ptbxl_database.csv"),
    index_col="ecg_id"
)

scp = pd.read_csv(
    os.path.join(DATA_PATH, "scp_statements.csv"),
    index_col=0
)

print("Number of ECG records:", len(df))
print("Number of columns:", len(df.columns))
print("Number of SCP statements:", len(scp))

Number of ECG records: 21799
Number of columns: 27
Number of SCP statements: 71


In [75]:
### 3.1 Dataset Structure

In [76]:
print("Columns:")
print(df.columns.tolist())

print("\nNumber of ECG records:", len(df))
print("Number of unique patients:", df["patient_id"].nunique())

Columns:
['patient_id', 'age', 'sex', 'height', 'weight', 'nurse', 'site', 'device', 'recording_date', 'report', 'scp_codes', 'heart_axis', 'infarction_stadium1', 'infarction_stadium2', 'validated_by', 'second_opinion', 'initial_autogenerated_report', 'validated_by_human', 'baseline_drift', 'static_noise', 'burst_noise', 'electrodes_problems', 'extra_beats', 'pacemaker', 'strat_fold', 'filename_lr', 'filename_hr']

Number of ECG records: 21799
Number of unique patients: 18869


In [77]:
### 3.2 SCP Diagnostic and Rhythm Codes

In [78]:
scp_codes = df["scp_codes"].apply(literal_eval)

code_counts = Counter()

for codes in scp_codes:
    code_counts.update(codes.keys())

print("Number of unique SCP codes:", len(code_counts))

print("\nMost common SCP codes:")

for code, count in code_counts.most_common(30):
    print(f"{code}: {count}")

Number of unique SCP codes: 71

Most common SCP codes:
SR: 16748
NORM: 9514
ABQRS: 3327
IMI: 2676
ASMI: 2357
LVH: 2132
NDT: 1825
LAFB: 1623
AFIB: 1514
ISC_: 1272
PVC: 1143
IRBBB: 1118
STD_: 1009
VCLVH: 875
STACH: 826
1AVB: 793
IVCD: 787
SARRH: 772
NST_: 767
ISCAL: 659
SBRAD: 637
QWAVE: 548
CRBBB: 541
CLBBB: 536
ILMI: 478
LOWT: 438
LAO/LAE: 426
NT_: 423
PAC: 398
AMI: 353


In [79]:
### 3.3 SCP Statement Definitions

In [80]:
print("Number of SCP statements:", len(scp))

print("\nColumns:")
print(scp.columns.tolist())

display(
    scp[
        [
            "description",
            "diagnostic",
            "form",
            "rhythm",
            "diagnostic_class",
            "diagnostic_subclass"
        ]
    ].head(20)
)

Number of SCP statements: 71

Columns:
['description', 'diagnostic', 'form', 'rhythm', 'diagnostic_class', 'diagnostic_subclass', 'Statement Category', 'SCP-ECG Statement Description', 'AHA code', 'aECG REFID', 'CDISC Code', 'DICOM Code']


,description,diagnostic,form,rhythm,diagnostic_class,diagnostic_subclass
NDT,non-diagnostic T abnormalities,1.0,1.0,NaN,STTC,STTC
NST_,non-specific ST changes,1.0,1.0,NaN,STTC,NST_
DIG,digitalis-effect,1.0,1.0,NaN,STTC,STTC
LNGQT,long QT-interval,1.0,1.0,NaN,STTC,STTC
NORM,normal ECG,1.0,NaN,NaN,NORM,NORM
IMI,inferior myocardial infarction,1.0,NaN,NaN,MI,IMI
ASMI,anteroseptal myocardial infarction,1.0,NaN,NaN,MI,AMI
LVH,left ventricular hypertrophy,1.0,NaN,NaN,HYP,LVH
LAFB,left anterior fascicular block,1.0,NaN,NaN,CD,LAFB/LPFB
ISC_,non-specific ischemic,1.0,NaN,NaN,STTC,ISC_


In [81]:
### 3.4 Diagnostic Classes

In [82]:
diagnostic_classes = scp[
    scp["diagnostic"] == 1
][
    ["description", "diagnostic_class", "diagnostic_subclass"]
]

display(diagnostic_classes)

,description,diagnostic_class,diagnostic_subclass
NDT,non-diagnostic T abnormalities,STTC,STTC
NST_,non-specific ST changes,STTC,NST_
DIG,digitalis-effect,STTC,STTC
LNGQT,long QT-interval,STTC,STTC
NORM,normal ECG,NORM,NORM
IMI,inferior myocardial infarction,MI,IMI
ASMI,anteroseptal myocardial infarction,MI,AMI
LVH,left ventricular hypertrophy,HYP,LVH
LAFB,left anterior fascicular block,CD,LAFB/LPFB
ISC_,non-specific ischemic,STTC,ISC_


In [83]:
print("Diagnostic classes:")
print(
    diagnostic_classes["diagnostic_class"]
    .value_counts(dropna=True)
)

Diagnostic classes:
diagnostic_class
MI      14
STTC    13
CD      11
HYP      5
NORM     1
Name: count, dtype: int64


In [84]:
### 3.5 Patient Distribution

In [85]:
ecgs_per_patient = df.groupby("patient_id").size()

print("Total ECG recordings:", len(df))
print("Unique patients:", df["patient_id"].nunique())

print("\nECGs per patient:")
print(ecgs_per_patient.describe())

print(
    "\nPatients with multiple ECGs:",
    (ecgs_per_patient > 1).sum()
)

print(
    "Maximum ECGs from one patient:",
    ecgs_per_patient.max()
)

Total ECG recordings: 21799
Unique patients: 18869

ECGs per patient:
count    18869.000000
mean         1.155281
std          0.523105
min          1.000000
25%          1.000000
50%          1.000000
75%          1.000000
max         10.000000
dtype: float64

Patients with multiple ECGs: 2111
Maximum ECGs from one patient: 10


In [86]:
### 3.6 Stratified Fold Structure

In [87]:
print("ECGs per fold:")
print(df["strat_fold"].value_counts().sort_index())

print("\nUnique patients per fold:")
print(
    df.groupby("strat_fold")["patient_id"]
      .nunique()
      .sort_index()
)

ECGs per fold:
strat_fold
1     2175
2     2181
3     2192
4     2174
5     2174
6     2173
7     2176
8     2173
9     2183
10    2198
Name: count, dtype: int64

Unique patients per fold:
strat_fold
1     1878
2     1849
3     1887
4     1883
5     1890
6     1884
7     1871
8     1881
9     1942
10    1904
Name: patient_id, dtype: int64


In [88]:
patient_fold_count = (
    df.groupby("patient_id")["strat_fold"]
      .nunique()
)

print(
    "Patients appearing in more than one fold:",
    (patient_fold_count > 1).sum()
)

print(
    "Maximum number of folds containing the same patient:",
    patient_fold_count.max()
)

Patients appearing in more than one fold: 0
Maximum number of folds containing the same patient: 1


In [89]:
### 3.7 Rhythm Annotations

In [90]:
rhythm_codes = scp[
    scp["rhythm"] == 1
]["description"]

rhythm_counts = {}

for code, count in code_counts.items():
    if code in rhythm_codes.index:
        rhythm_counts[code] = count

for code, count in sorted(
    rhythm_counts.items(),
    key=lambda x: x[1],
    reverse=True
):
    description = scp.loc[code, "description"]
    print(
        f"{code:6s} "
        f"{count:6,d} "
        f"{description}"
    )

SR     16,748 sinus rhythm
AFIB    1,514 atrial fibrillation
STACH     826 sinus tachycardia
SARRH     772 sinus arrhythmia
SBRAD     637 sinus bradycardia
PACE      294 normal functioning artificial pacemaker
SVARR     157 supraventricular arrhythmia
BIGU       82 bigeminal pattern (unknown origin, SV or Ventricular)
AFLT       73 atrial flutter
SVTAC      27 supraventricular tachycardia
PSVT       24 paroxysmal supraventricular tachycardia
TRIGU      20 trigeminal pattern (unknown origin, SV or Ventricular)


In [91]:
### 3.8 Number of SCP Labels per ECG

In [92]:
label_counts = df["scp_codes"].apply(
    lambda x: len(literal_eval(x))
)

print("Number of SCP labels per ECG:")
print(label_counts.describe())

print("\nExact distribution:")
print(label_counts.value_counts().sort_index())

print(
    "\nPercentage of ECGs with multiple labels:",
    (label_counts > 1).mean() * 100
)

Number of SCP labels per ECG:
count    21799.000000
mean         2.798615
std          1.211884
min          1.000000
25%          2.000000
50%          2.000000
75%          3.000000
max          9.000000
Name: scp_codes, dtype: float64

Exact distribution:
scp_codes
1      709
2    11225
3     5101
4     2594
5     1251
6      596
7      253
8       63
9        7
Name: count, dtype: int64

Percentage of ECGs with multiple labels: 96.74755722739576


In [93]:
### 3.9 Common SCP Label Combinations

In [94]:
from collections import Counter

combination_counts = Counter()

for codes in df["scp_codes"].apply(literal_eval):
    combination = tuple(sorted(codes.keys()))
    combination_counts[combination] += 1

print("Most common SCP label combinations:\n")

for combination, count in combination_counts.most_common(30):
    print(
        f"{' + '.join(combination):30s}: {count:,}"
    )

Most common SCP label combinations:

NORM + SR                     : 7,062
NDT + SR                      : 633
ABQRS + IMI + SR              : 496
NORM + SARRH                  : 339
LAFB + SR                     : 298
NORM + SBRAD                  : 294
PACE                          : 273
CLBBB + SR                    : 212
LVH + SR + VCLVH              : 193
NORM                          : 190
IRBBB + SR                    : 189
ABQRS + NORM + SR             : 178
NORM + STACH                  : 176
IRBBB + NORM + SR             : 176
IMI + SR                      : 174
ISC_ + LVH + SR               : 163
ABQRS + ASMI + SR             : 162
NST_ + SR                     : 154
NDT + NT_ + SR                : 123
ABQRS + ASMI + IMI + SR       : 118
LVH + SR                      : 103
AFIB + NST_                   : 97
ABQRS + ILMI + SR             : 93
NORM + PVC + SR               : 88
AFIB + NDT                    : 87
NORM + SR + VCLVH             : 84
ASMI + SR                     

In [95]:
### 3.10 Annotation and Validation Metadata

In [96]:
validation_columns = [
    "validated_by",
    "second_opinion",
    "validated_by_human",
    "initial_autogenerated_report"
]

for column in validation_columns:
    print(f"\n===== {column} =====")
    print(df[column].value_counts(dropna=False))


===== validated_by =====
validated_by
NaN     9378
0.0     6120
1.0     5127
2.0      566
3.0      175
4.0      115
6.0      100
5.0       90
7.0       75
8.0       37
9.0        9
10.0       6
11.0       1
Name: count, dtype: int64

===== second_opinion =====
second_opinion
False    21244
True       555
Name: count, dtype: int64

===== validated_by_human =====
validated_by_human
True     16056
False     5743
Name: count, dtype: int64

===== initial_autogenerated_report =====
initial_autogenerated_report
False    14986
True      6813
Name: count, dtype: int64


In [97]:
### 3.11 ECG Quality Annotations

In [98]:
quality_columns = [
    "baseline_drift",
    "static_noise",
    "burst_noise",
    "electrodes_problems",
    "extra_beats",
    "pacemaker"
]

quality_summary = []

for column in quality_columns:
    count = df[column].notna().sum()
    percentage = count / len(df) * 100

    quality_summary.append({
        "annotation": column,
        "ECGs": count,
        "percentage": percentage
    })

quality_summary = pd.DataFrame(quality_summary)

display(quality_summary)

,annotation,ECGs,percentage
0,baseline_drift,1598,7.330611
1,static_noise,3260,14.954814
2,burst_noise,613,2.812056
3,electrodes_problems,30,0.137621
4,extra_beats,1949,8.940777
5,pacemaker,291,1.334924


In [99]:
### 3.12 Patient Independence of Stratified Folds

In [100]:
patient_fold_counts = (
    df.groupby("patient_id")["strat_fold"]
      .nunique()
)

multi_fold_patients = (
    patient_fold_counts > 1
).sum()

print(
    "Patients appearing in more than one fold:",
    multi_fold_patients
)

print(
    "Maximum folds containing the same patient:",
    patient_fold_counts.max()
)

Patients appearing in more than one fold: 0
Maximum folds containing the same patient: 1


In [101]:
## 3.13 Exploration Summary

#The PTB-XL dataset contains 21,799 ECG recordings from 18,869 unique patients.

#Key observations:

#- The dataset contains 71 SCP-ECG statements.
#- ECGs commonly contain multiple simultaneous annotations.
#- Approximately 96.7% of ECGs have more than one SCP label.
#- Multiple diagnostic and rhythm findings can coexist within the same ECG.
#- Five major diagnostic classes are available for our downstream research:
#  NORM, MI, STTC, CD, and HYP.
#- The dataset contains substantial rhythm and signal-quality annotations.
#- Some ECGs contain baseline drift, static noise, burst noise, extra beats,
#  and other quality issues.
#- The provided strat_folds are patient-independent: no patient appears
#  in more than one fold.

#These characteristics motivate a multilabel ECG classification framework
#with explicit patient-independent evaluation and later robustness analysis.